# Ablation Study — Stage 2: Regime Forecasting

**Experimentos:** B1 (Complexidade do Modelo) · B2 (Feature Engineering)

Avalia o quanto as escolhas de design no Stage 2 (previsão de regime)
impactam o desempenho geral do pipeline, em comparação com o Stage 1.

Além de `ADD`, este notebook passa a reportar métricas de classificação de estado
nas versões original e balanceada, para reduzir a distorção causada pelo
 desbalanceamento entre regimes bull e bear.

Neste contexto, `State Accuracy` deve ser lida como concordância com o pseudo-ground-truth
consensual de regimes, enquanto `State Balanced Accuracy` é a medida preferencial quando se
quer comparar o desempenho entre classes bull e bear sem deixar a classe majoritária dominar a métrica.

**Hipótese central:** O modelo de previsão contribui menos para a variância
total do que as escolhas de identificação de regime (Stage 1).

**JIT:** Métricas calculadas com Numba  
**Polars:** Análise e agregação de resultados

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.config.settings import (
    ASSETS, TEST_START, TEST_END,
    ASSET_TICKERS, DATA_START, DATA_END, FRED_SERIES,
)

from src.ablation.ablation_runner import (
    run_single_ablation, prepare_ablation_data,
    ABLATION_B1_CONFIGS, ABLATION_B2_CONFIGS,
    analyze_ablation, compare_ablations,
)
from src.ablation.statistical_tests import pairwise_comparison_table, cohens_d
from src.ablation.polars_utils import build_ablation_summary, float_nan_to_null
from src.ablation.jit_metrics import compute_metrics_array

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✓ Imports concluídos')

In [ ]:
# Carrega dados
loader = DataLoader(cache_dir=str(ROOT / 'data' / 'raw'))
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred_raw = loader.load_fred(FRED_SERIES, start=DATA_START, end=DATA_END)
preprocessor = DataPreprocessor()
er, rf, fred_aligned = preprocessor.prepare(prices, fred_raw)

DEMO_ASSETS = ASSETS
N_SEEDS = 20

# Pré-computa dados
features_cache = {}
vol_cache = {}
true_regimes_cache = {}

print('Preparando dados...')
for asset in tqdm(DEMO_ASSETS):
    ohlc, features, vol_estimators, true_regimes = prepare_ablation_data(
        asset=asset, er=er, rf=rf, fred=fred_aligned
    )
    features_cache[asset]     = features
    vol_cache[asset]          = vol_estimators
    true_regimes_cache[asset] = true_regimes

print('✓ Dados prontos')

## 1. Ablation B1 — Complexidade do Modelo de Previsão

Compara seis modelos de crescente complexidade:

| Modelo | Parâmetros | Interpretabilidade |
|--------|------------|--------------------|
| Persistence (s_{t+1}=s_t) | 0 | Perfeita |
| Logistic Regression | ~20 | Alta |
| Decision Tree (depth≤3) | ~30 | Alta |
| Decision Tree (depth≤5) | ~60 | Média |
| Random Forest (100 trees) | ~3000 | Baixa |
| XGBoost (default) | ~500 | Baixa |

In [ ]:
B1_RESULTS_RAW = []

print(f'Ablation B1: {len(ABLATION_B1_CONFIGS)} modelos × {len(DEMO_ASSETS)} ativos × {N_SEEDS} seeds')
t0 = time.time()

for asset in tqdm(DEMO_ASSETS, desc='Assets'):
    for config in tqdm(ABLATION_B1_CONFIGS, desc=f'Modelos B1 ({asset})', leave=False):
        for seed in range(N_SEEDS):
            res = run_single_ablation(
                config         = config,
                asset          = asset,
                er             = er,
                rf             = rf,
                features       = features_cache[asset],
                vol_estimators = vol_cache[asset],
                true_regimes   = true_regimes_cache[asset],
                seed           = seed,
            )
            row = res.to_dict()
            # Adiciona metadados do modelo
            model_meta = {
                'persistence': {'params': 0,    'interpretability': 'Perfeita'},
                'logistic':    {'params': 20,   'interpretability': 'Alta'},
                'tree_d3':     {'params': 30,   'interpretability': 'Alta'},
                'tree_d5':     {'params': 60,   'interpretability': 'Média'},
                'rf_100':      {'params': 3000, 'interpretability': 'Baixa'},
                'xgb_default': {'params': 500,  'interpretability': 'Baixa'},
            }
            if config.name in model_meta:
                row.update(model_meta[config.name])
            B1_RESULTS_RAW.append(row)

B1_DF = float_nan_to_null(pl.DataFrame(B1_RESULTS_RAW))
print(f'✓ B1 concluído em {time.time() - t0:.1f}s')

In [ ]:
# Sumário B1 com ordem de complexidade
MODEL_ORDER = ['persistence', 'logistic', 'tree_d3', 'tree_d5', 'rf_100', 'xgb_default']

B1_SUMMARY = (
    B1_DF
    .group_by('config')
    .agg([
        pl.col('add').mean().alias('ADD_mean'),
        pl.col('add').std().alias('ADD_std'),
        pl.col('state_accuracy').mean().alias('StateAccuracy'),
        pl.col('state_balanced_accuracy').mean().alias('StateBalAccuracy'),
        pl.col('f1_score').mean().alias('F1'),
        pl.col('sortino_ratio').mean().alias('Sortino_mean'),
        pl.col('sortino_ratio').std().alias('Sortino_std'),
        pl.col('training_time_seconds').mean().alias('Train_time_s'),
    ])
    .with_columns([
        pl.col('config').map_elements(
            lambda x: MODEL_ORDER.index(x) if x in MODEL_ORDER else 99,
            return_dtype=pl.Int32,
        ).alias('order')
    ])
    .sort('order')
    .drop('order')
)

print('Tabela B1: Complexidade do Modelo')
print('=' * 80)
print(B1_SUMMARY)

In [ ]:
# Visualização B1: Performance vs. Complexidade
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
b1_pd = B1_SUMMARY.to_pandas().set_index('config')
b1_pd = b1_pd.reindex([m for m in MODEL_ORDER if m in b1_pd.index])

x = range(len(b1_pd))
labels = b1_pd.index.tolist()

# (a) ADD
axes[0,0].bar(x, b1_pd['ADD_mean'], yerr=b1_pd['ADD_std'],
              color='steelblue', alpha=0.8, capsize=4, edgecolor='white')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(labels, rotation=30, ha='right')
axes[0,0].set_ylabel('ADD (dias)'); axes[0,0].set_title('(a) Detection Delay por Modelo')
baseline_add = b1_pd.loc['xgb_default', 'ADD_mean'] if 'xgb_default' in b1_pd.index else None
if baseline_add:
    axes[0,0].axhline(baseline_add, color='red', linestyle='--', alpha=0.6, label='XGBoost baseline')
    axes[0,0].legend(fontsize=8)

# (b) Sortino
axes[0,1].bar(x, b1_pd['Sortino_mean'], yerr=b1_pd['Sortino_std'],
              color='darkorange', alpha=0.8, capsize=4, edgecolor='white')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(labels, rotation=30, ha='right')
axes[0,1].set_ylabel('Sortino Ratio'); axes[0,1].set_title('(b) Sortino por Modelo')

# (c) Accuracy original, balanceada e F1
width = 0.25
axes[1,0].bar([i - width for i in x], b1_pd['StateAccuracy'], width,
              label='State Accuracy', color='#2ecc71', alpha=0.8, edgecolor='white')
axes[1,0].bar(x, b1_pd['StateBalAccuracy'], width,
              label='State Balanced Accuracy', color='#3498db', alpha=0.8, edgecolor='white')
axes[1,0].bar([i + width for i in x], b1_pd['F1'], width,
              label='F1-Score', color='#9b59b6', alpha=0.8, edgecolor='white')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(labels, rotation=30, ha='right')
axes[1,0].set_ylabel('Score'); axes[1,0].set_title('(c) Métricas de Classificação')
axes[1,0].legend(fontsize=8)

# (d) Trade-off: Sortino vs. Training time
axes[1,1].scatter(b1_pd['Train_time_s'], b1_pd['Sortino_mean'],
                  s=100, alpha=0.8, color='#e74c3c', zorder=5)
for idx, (name, row) in enumerate(b1_pd.iterrows()):
    axes[1,1].annotate(name, (row['Train_time_s'], row['Sortino_mean']),
                       textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[1,1].set_xlabel('Tempo de Treino (s)')
axes[1,1].set_ylabel('Sortino Ratio')
axes[1,1].set_title('(d) Complexidade-Performance Trade-off')

plt.suptitle('Ablation B1: Complexidade do Modelo de Previsão', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'ablation_B1_model_complexity.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Ablation B2 — Feature Engineering

| Feature Set | Descrição | # Features |
|-------------|-----------|------------|
| Minimal | Apenas retornos com lag | ~5 |
| Standard | Retornos + volatilidade | ~13 |
| Extended | Standard + momentum | ~20 |
| Kitchen Sink | Todas disponíveis | ~25 |

In [ ]:
B2_RESULTS_RAW = []

print(f'Ablation B2: {len(ABLATION_B2_CONFIGS)} feature sets × {len(DEMO_ASSETS)} ativos × {N_SEEDS} seeds')
t0 = time.time()

for asset in tqdm(DEMO_ASSETS, desc='Assets'):
    for config in ABLATION_B2_CONFIGS:
        for seed in range(N_SEEDS):
            res = run_single_ablation(
                config         = config,
                asset          = asset,
                er             = er,
                rf             = rf,
                features       = features_cache[asset],
                vol_estimators = vol_cache[asset],
                true_regimes   = true_regimes_cache[asset],
                seed           = seed,
            )
            B2_RESULTS_RAW.append(res.to_dict())

B2_DF = float_nan_to_null(pl.DataFrame(B2_RESULTS_RAW))
print(f'✓ B2 concluído em {time.time() - t0:.1f}s')

B2_SUMMARY = (
    B2_DF
    .group_by('config')
    .agg([
        pl.col('add').mean().alias('ADD_mean'),
        pl.col('state_accuracy').mean().alias('StateAccuracy'),
        pl.col('state_balanced_accuracy').mean().alias('StateBalAccuracy'),
        pl.col('sortino_ratio').mean().alias('Sortino_mean'),
        # Overfitting proxy: gap entre accuracy de treino e teste
        pl.col('regime_ari').std().alias('ARI_var'),
    ])
    .sort('config')
)
print('\nTabela B2: Feature Engineering')
print(B2_SUMMARY)

In [ ]:
# Análise estatística B1 vs B2: qual Stage 2 choice tem mais impacto?
B1_ANALYSIS = analyze_ablation(B1_DF.with_columns(pl.lit('B1').alias('ablation_id')), 'add')
B2_ANALYSIS = analyze_ablation(B2_DF.with_columns(pl.lit('B2').alias('ablation_id')), 'add')

print('B1 — Friedman p-value:', round(B1_ANALYSIS['friedman']['p_value'], 4))
print('B2 — Friedman p-value:', round(B2_ANALYSIS['friedman']['p_value'], 4))

print('\nVariância explicada:')
print(f"  B1 (Modelo):    {B1_ANALYSIS['variance_decomp']['config_contribution']:.1%}")
print(f"  B2 (Features):  {B2_ANALYSIS['variance_decomp']['config_contribution']:.1%}")

# Salva resultados
results_dir = ROOT / 'results' / 'ablation'
results_dir.mkdir(parents=True, exist_ok=True)
B1_DF.write_parquet(results_dir / 'ablation_B1.parquet')
B2_DF.write_parquet(results_dir / 'ablation_B2.parquet')
print('\n✓ Resultados salvos em Parquet')

## Conclusões — Stage 2

| Achado | Descrição |
|--------|----------|
| **B1 (Modelo)** | O ganho de acurácia favorece modelos simples. `logistic` obteve a melhor combinação de `State Accuracy` (`0.620`) e `State Balanced Accuracy` (`0.673`), superando `xgb_default` (`0.608` e `0.633`) e também produzindo o menor `ADD` (`25.35`). |
| **B1 (Interpretação)** | A diferença entre `State Accuracy` e `State Balanced Accuracy` é sistematicamente positiva, indicando que a acurácia balanceada recompensa melhor a identificação do regime minoritário. `persistence` ilustra isso bem: apesar de `State Accuracy = 0.575`, o baixo `F1` (`0.453`) e o `ADD` elevado (`56.15`) mostram que o modelo falha em capturar transições relevantes. |
| **B2 (Features)** | `minimal` reduz a acurácia bruta (`0.581` vs `0.608` do `standard`), mas eleva ligeiramente a `State Balanced Accuracy` (`0.638` vs `0.633`). Isso sugere que um conjunto enxuto pode melhorar o equilíbrio entre classes, embora perca aderência global ao pseudo-ground-truth. |
| **B2 (Plateau)** | `standard`, `extended` e `kitchen_sink` produzem métricas praticamente idênticas (`State Accuracy ≈ 0.608`, `State Balanced Accuracy ≈ 0.633`, `F1 ≈ 0.500`). Na prática, isso indica saturação de informação: adicionar features além do conjunto base não trouxe ganho detectável de classificação. |
| **Stage 1 vs Stage 2** | As escolhas do Stage 1 continuam mais determinantes para a qualidade da detecção; no Stage 2, a principal conclusão sobre acurácia é que complexidade adicional não garantiu melhor classificação de regime. |

**Síntese metodológica.** No Stage 2, `State Balanced Accuracy` é a métrica mais adequada para comparar classificadores sob desbalanceamento entre bull e bear. A leitura conjunta com `F1` e `ADD` evita superestimar modelos que acertam a classe majoritária, mas falham em antecipar regimes menos frequentes.

**Próximo:** Notebook 09 — Stage 3 (Portfolio Allocation)